# Osaka Geospatial AI — Google Colab

このNotebookは公開GitHubリポジトリを取得し、依存関係、国土数値情報、Hugging Faceモデルを自動取得して、パイプラインと成果物表示を行います。GIS・ML・VLM・LLMの処理ロジックはNotebookではなく `src/` と `scripts/` にあります。

**実行前に Colab の `ランタイム` → `ランタイムのタイプを変更` → `T4 GPU` を選択し、`すべてのセルを実行`してください。** 初回実行は公開GISデータとモデルのダウンロードを含むため時間がかかります。

## 1. Environment Setup

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Mr-Kondo/osaka-geospatial-ai.git"
REPO_REF = "main"
IN_COLAB = "google.colab" in sys.modules
ROOT = Path("/content/osaka-geospatial-ai") if IN_COLAB else (
    Path.cwd() if Path("pyproject.toml").exists() else Path.cwd().parent
)
HF_HOME = Path("/content/huggingface") if IN_COLAB else Path.home() / ".cache/huggingface"
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print(f"Colab: {IN_COLAB}")
print(f"Repository: {REPO_URL}@{REPO_REF}")
print(f"Hugging Face cache: {HF_HOME}")

## 2. Clone / Install

ColabではGitHubの `main` をcloneします。セルを再実行した場合はfast-forward可能な更新だけを取得します。依存関係にはGIS、ML、Transformers、Qwen VLM/LLM実行環境が含まれます。

In [ ]:
if IN_COLAB:
    if not ROOT.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(ROOT)],
            check=True,
        )
    elif (ROOT / ".git").exists():
        subprocess.run(["git", "-C", str(ROOT), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
        subprocess.run(["git", "-C", str(ROOT), "checkout", REPO_REF], check=True)
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only", "origin", REPO_REF], check=True)
    else:
        raise RuntimeError(f"{ROOT} exists but is not a Git repository")

os.chdir(ROOT)
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[ai]"], check=True)

from IPython.display import HTML, JSON, Image, Markdown, display
from osaka_geo_ai.presentation import (
    configuration_summary,
    dataset_summary,
    metrics_table,
    prediction_preview,
)

## 3. Configuration

`configs/colab.yaml` はオンラインデータ取得とHugging Face VLM/LLMを有効にします。この節は読み取り専用のPresentation adapterが設定と実行環境を表示します。

In [ ]:
display(configuration_summary(ROOT, "configs/colab.yaml"))

## 4. Run Pipeline

このセルがGitHubから取得したPython scriptを実行します。初回は国土交通省の公開データとHugging Faceモデルを自動取得します。後続処理は明示的なParquet/JSON/PNG/Markdown成果物を介して進みます。

In [ ]:
subprocess.run(
    [sys.executable, "scripts/run_pipeline.py", "--config", "configs/colab.yaml"],
    cwd=ROOT,
    check=True,
)

## 5. Dataset Summary

人口は2010年国勢調査を基準にした将来推計を含み、現在人口の観測値ではありません。

In [ ]:
display(dataset_summary(ROOT))

## 6. Interactive GIS Map

探索用HTMLです。FoliumのJavaScript/CSS読込にはインターネット接続が必要です。

In [ ]:
display(HTML(filename=str(ROOT / "artifacts/maps/land_price_map.html")))

## 7. Static Maps

観測値・予測・残差を区別してください。正の残差は過小予測、負は過大予測です。4面比較画像がVLM入力です。

In [ ]:
display(Image(filename=str(ROOT / "artifacts/figures/land_price_map.png"), width=650))
display(Image(filename=str(ROOT / "artifacts/figures/prediction_map.png"), width=650))
display(Image(filename=str(ROOT / "artifacts/figures/vlm_input_overview.png"), width=1000))

## 8. Model Metrics

モデル選択はvalidation MAEを用います。`mae_pp` と `rmse_pp` はパーセントポイントです。

In [ ]:
display(metrics_table(ROOT))

## 9. Prediction Results

2025年テスト予測の抜粋です。別途、2026年予測を `forecast.parquet` に保存します。

In [ ]:
display(prediction_preview(ROOT))

## 10. VLM Analysis

Hugging Face VLMによる視覚的観察です。正確な数値計算や因果推論として扱わず、`status` と `limitations` を確認してください。

In [ ]:
display(JSON(filename=str(ROOT / "artifacts/vlm/map_analysis.json")))

## 11. Final Report

検証済み `analysis.json` から作る定型説明に、Hugging Face LLMの生成説明を区別して追記します。

In [ ]:
display(JSON(filename=str(ROOT / "artifacts/reports/report_generation.json")))
display(Markdown((ROOT / "artifacts/reports/report.md").read_text(encoding="utf-8")))